# N-Gram Sprachmodelle

N-Gram Sprachmodelle sind probabilistische Modelle, die Sequenzen von Token modellieren. Auch wenn sie nicht mit modernen Sprachmodellen mithalten können, bilden sie doch das historische und konzeptionelle Fundament der modernen Sprachmodellierung. Anhand der N-Gram-basierten Sprachmodellierung lassen sich sämtliche Kernkonzepte der statistischen Sprachmodellierung einfach erklären: **Tokenisierung**, **Wahrscheinlichkeitsverteilungen über diskrete Zustandsräume** und die **Vorhersage des nächsten Tokens gegeben eines Kontexts**.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/workshop-bundeskonferenz"):
        !git clone -q --depth 1 https://github.com/VoErik/workshop-bundeskonferenz.git /content/workshop-bundeskonferenz
    %cd /content/workshop-bundeskonferenz
    %pip install -q .

## Datensatz

Als Beispieldatensatz verwenden wir hier den `tiny-shakespeare` Datensatz. Dieser enthält die (meisten) Werke von William Shakespeare in Form einer einfachen `.txt` Datei. Damit diese Daten für die N-Gram Modellierung verwendbar sind, müssen sie zuerst aufbereitet werden:

In [ ]:
DATENSATZ = "assets/tiny_shakespeare.txt"
LEVEL = "char" # "char" (Zeichen) oder "word" (Wort)

1. Laden der Daten: Aus der `.txt`-Datei in einen `String`

```python
data = lade_datensatz(source="assets/tiny_shakespeare.txt")
```

2. Erstellen der Trainingsbeispiele (Chunking): In diesem Fall $\rightarrow$ ein Sprechakt = ein Trainingsbeispiel
3. Tokenisierung: Umwandeln der Sprechakte in Tokens

```python
chunks = [tokenisiere(c, level) for c in als_chunks(text=data, separator="\n\n") if c.strip()]
```

Die `bereite_daten_vor()` Funktion übernimmmt genau diese Schritte:

In [ ]:
from workshop import bereite_daten_vor

daten, vokabular = bereite_daten_vor(
    datensatz=DATENSATZ,
    level=LEVEL,
)

print(f"Anzahl Trainingsbeispiele: {len(daten)}")
print(f"Erstes Trainingsbeispiel: \n{daten[0]}")

Das Vokabular (`vocabulary`) $V$ ist die Menge der einzigarten Token, die im Trainingsdatensatz vorkommen. Bei einem Modell auf Zeichenebene umfasst das alle Buchstaben, Ziffern und Sonderzeichen, bei einem Modell auf Wortebene alle einzigartigen Wörter.

Die Größe des Vokabulars $\mathopen| V \mathclose|$ beschreibt den *Zustandsraum* des Modells. Token, die nicht im Vokabular enthalten sind kennt das Modell nicht und können nicht verarbeitet werden.

Die folgende Codezelle zeigt das Vokabular (in seiner Gänze für ein Zeichen-Level Modell bzw. die ersten und letzten 100 Einträge für das Wort-Level Modell):

In [ ]:
if LEVEL == "char":
    print(f"Vokabular:\n{vokabular}")
else:
    print(f"Vokabular:\n{vokabular[:100]} ... \n\n ... {vokabular[-100:]}")
print(f"Anzahl Vokabulareinträge: {len(vokabular)}")

## Das N-Gram Modell

Ein **Sprachmodell** berechnet die Wahrscheinlichkeit $P$ einer Sequenz von Tokens $w_1, \dots, w_T$:

$$P(w_1, \dots, w_T) = \prod_{i=1}^T P(w_i \mid w_1, \dots, w_{i-1})$$

Für lange Tokensequenzen wird diese Berechnung unmöglich. Daher treffen N-Gram Sprachmodelle die **Markov-Annahme**: die Wahrscheinlichkeit des Tokens $i$ hängt nur von den vorherigen $N-1$ Tokens ab, wobei $N$ als **Ordnung** des N-Gram Modells bezeichnet wird:

$$P(w_i \mid w_1, \dots, w_{i-1} \approx P(w_i \mid w_{i-(N-1)}, \dots, w_{i-1})$$

In [ ]:
from workshop import NGram

ngram = NGram(
    n=2,        # Ordnung des Modells
    level=LEVEL # "char" (Zeichen) oder "word" (Wort)
).fit(docs=daten, vokabular=vokabular)

## Generieren neuer Token

Nachdem das Modell auf den Daten "trainiert" wurde können wir es verwenden, um neue Daten zu generieren. Um zu verstehen, wie ein N-Gram Modell neue Daten generiert, hilft es, sich das Modell als Übergangsmatrix vorzustellen:

* Jede Zeile dieser Matrix repräsentiert den *Kontext* $\rightarrow$ die letzten $N-1$ Token.
* Jede Spalte dieser Matrix repräsentiert das mögliche Folgetoken.
* Der Wert in einer Zelle ist die *Übergangswahrscheinlichkeit*: $P(\texttt{Folgetoken} \mid \texttt{Kontext})$.
* Die Summe aller Übergangswahrscheinlichkeiten einer Zeile ist $1$.
* Während der **Inferenz** wird aus dieser Matrix gesamplet.

<details>
<summary>🎓 <b>Gesetz der großen Zahlen & Bedingte Wahrscheinlichkeiten</b></summary>

**Bedingte Wahrscheinlichkeiten**: Die Berechnung der N-Gram-Wahrscheinlichkeiten ist eine direkte Anwendung der *Bedingten Wahrscheinlichkeit* $P(A \mid B)$.

**Gesetz der großen Zahlen**: Das Training des Modells veranschaulicht das empirische Gesetz der großen Zahlen: Wir schätzen die wahre, unsichtbare Wahrscheinlichkeit eines Folgetokens, indem wir einfach seine *relative Häufigkeit* in einem großen Textkorpus ermitteln.

**Zufallsexperimente**: Das Generieren des nächsten Tokens ist ein Zufallsexperiment. Im Gegensatz zu einem fairen Würfel (Gleichverteilung) ziehen wir hier aus einer *gewichteten Verteilung*. Das Modell "würfelt" das nächste Zeichen, wobei Zeichen mit hoher relativer Häufigkeit eine entsprechend größere Trefferfläche auf unserem gedachten "Sprach-Würfel" haben.

</details>

In [ ]:
LAENGE = 500 # wie viele neue Token generiert werden sollen
PROMPT = ""  # Startkontext

_ = ngram.generiere(LAENGE, PROMPT)

## N-Gram Visualisierung

Die folgende Codezelle visualisiert die Übergangsmatrix für unser N-Gram Modell:

In [ ]:
ngram.zeige_assoziations_matrix()

<details>
<summary>🎓 <b>Lineare Algebra, Graphentheorie, Markov-Ketten</b></summary>

Die Assoziationsmatrix ist formal eine *Übergangsmatrix* (Stochastische Matrix), wie sie bei Markov-Ketten verwendet wird (die Summe jeder Zeile ist exakt 1). Das lässt sich visualisieren, indem man die Kontexte als Knoten und die Übergangswahrscheinlichkeiten als Kanten eines *gerichteten Graphen* zeichnet.

</details>

## Probleme

### Skalierbarkeit - Curse of Dimensionality

Die Anzahl der möglichen N-Gramme wächst exponentiell mit der Größe des Vokabulars $\mathopen| {V} \mathclose|$ und *Ordnung* $N$. Insgesamt gibt es $\mathopen| {V} \mathclose|^N$ Kombinationen.

---

**Beispiel:**

Für ein einfaches Charakter-Level Bi-Gram ohne Sonderzeichen und Groß- und Kleinschreibung ergibt sich eine Matrix der Größe:
$$ \mathopen| {V} \mathclose|^N = 26^2 = 676$$

Für ein modernes Sprachmodell wie GPT-4, liegt die Größe des Vokabulars bei $200.000$. Bereits bei einem Bi-Gram würde sich damit eine Matrix der Größe:

$$ \mathopen| {V} \mathclose|^N = 200.000^2 = 4 \times 10^{10}$$

ergeben.

---

<details>
<summary>🎓 <b>Kombinatorik, Exponentialfunktion</b></summary>

Der **Fluch der Dimensionalität** veranschaulicht exponentielles Wachstum. Bei einem Vokabular der Größe $\mathopen| V \mathclose|$ gibt es für ein N-Gram-Modell exakt $\mathopen| V \mathclose|^N$ mögliche Kombinationen (Ziehen mit Zurücklegen unter Beachtung der Reihenfolge).

</details>

Die folgende Codezelle zeigt, wie die Größe der Matrix in Abhängigkeit von Vokabular und Ordnung des Modells wächst:

In [ ]:
from workshop.viz import plot_ngram_matrix_groessen

vokabular_groessen = (5, 10, 20, 40)
n_gramme = (2, 3, 4, 5, 6, 7, 8)
n_trainingsdaten = sum(len(doc) for doc in daten)

_ = plot_ngram_matrix_groessen(
    vokabular_groessen=vokabular_groessen,
    n_gramme=n_gramme,
    n_trainingsdaten=n_trainingsdaten,
)

### Sparsity

Die gestrichelte graue Linie zeigt ein weiteres Problem auf: Die Größe der Matrix übersteigt die Anzahl der Trainingstokens. Konkret heißt das: eine gewisse Anzahl der möglichen $\mathopen| V \mathclose|^N$ Kombinationen existiert in den Trainingsdaten nicht.

> <b>Frage</b>: Was passiert, wenn während der Generierung neuer Token eine solche unbekannte Sequenz auftritt?

<details>
<summary>✅ <b>Antwort</b></summary>

Für solche Kombinationen beträgt die Übergangswahrscheinlichkeit $P = 0$. Tritt während der Inferenz eine solche Kombination auf beträgt die Wahrscheinlichkeit der gesamten Sequenz entsprechend auch $0$!

</details>

> <b>Frage</b>: Wie kann dieses Problem verhindert werden?

<details>
<summary>✅ <b>Antwort</b></summary>

In der Praxis werden Techniken wie **Laplace-Smoothing** verwendet, bei denen die Wahrscheinlichkeitsmasse von gesehenen Ereignissen (Ereignissen aus den Trainingsdaten) auf ungesehene Ereignisse (Ereignisse nicht in den Trainingsdaten) umverteilt werden:

Angenommen, die Buchstabenkombination "qr" kam in den Trainingsdaten nie vor, dann ist ihre Wahrscheinlichkeit: $P(r \mid q) = 0$

**Laplace-Smoothing** (oder *additive Glättung*) löst dieses Problem, indem jedem Eintrag der Übergangsmatrix der gleiche konstante Wert (z.B. $1$) hinzugefügt wird. Damit ergibt sich eine neue Übergangswahrscheinlichkeit $P(r \mid q)$:

$$ P(r \mid q) = \frac{\texttt{Alte Anzahl (q} \rightarrow \texttt{r)} + 1}{\texttt{Alte Zeilensumme q} + 26} $$

</details>

### Gedächtnisverlust

Ein N-Gram Modell hat ein Kontextfenster (Gedächtnis) von $N-1$. Eine Frage wie:

$$\texttt{John und Mary sitzen am Tisch. Mit wem sitzt John am Tisch? ...}$$

würden Sie vermutlich problemlos mit "Mary" beantworten. Ein N-Gram Modell geringer Ordnung hat an der Stelle jedoch leider wieder "vergessen" dass es Mary jemals gab. N-Gram Sprachmodelle sind also nicht in der Lage, Informationen über längere Kontexte zu erhalten.

## Anwendungsbeispiele

Trotz der Limitationen waren und sind N-Gram Modelle wegen ihrer Einfachheit sehr beliebt und werden beispielsweise in Smartphones für die Autokorrektur oder Autovervollständigung verwendet.

---

**Beispiel: Kontextbasierte Autokorrektur**

Ein Nutzer tippt auf dem Smartphone: $\texttt{Ich trinke eine Tasse Twe}$.

Das System registriert $\texttt{Twe}$ als Out-of-Vocabulary (OOV) und generiert über eine Metrik für Tasten-Distanzen (z.B. Levenshtein) mögliche Korrekturkandidaten aus dem Vokabular: $\texttt{Tee}$ und $\texttt{Two}$.

Um zu entscheiden, welches Wort gemeint war, berechnet ein Bi-Gram-Modell die Wahrscheinlichkeit der Kandidaten gegeben den Kontext (das vorherige Token $\texttt{Tasse}$):

$$P(\text{Tee} | \text{Tasse}) = \frac{C(\text{Tasse, Tee})}{C(\text{Tasse})} = 0.18$$
$$P(\text{Two} | \text{Tasse}) = \frac{C(\text{Tasse, Two})}{C(\text{Tasse})} = 0.00001$$

Obwohl beide Wörter gültige Einträge im Systemvokabular sein können, ist die Übergangswahrscheinlichkeit für "Tee" deutlich höher. Das System korrigiert die Eingabe automatisch zu $\texttt{Tasse Tee}$.

---

**Beispiel: T9-Tastaturen**

Auf numerischen Handytastaturen waren mehrere Buchstaben auf einer Taste gruppiert. Die Eingabe der Ziffernfolge **4-2-8-7** (4=GHI, 2=ABC, 8=TUV, 7=PQRS) erzeugt theoretisch $3 \times 3 \times 3 \times 4 = 108$ mögliche Zeichenkombinationen.

Verwendet das System ein Bi-Gram-Modell und der Nutzer hat zuvor $\texttt{aus dem}$ getippt, berechnet das System die Wahrscheinlichkeit für alle 108 Kombinationen gegeben den Kontext $\texttt{dem}$. Da $P(\texttt{Haus} \mid \texttt{dem}) \gg P(\texttt{Haut} \mid \texttt{dem})$, wird $\texttt{Haus}$ als primärer Treffer auf dem Display gezeigt, ohne man manuell durch die Optionen blättern muss.

<details>
<summary> Anmerkung </summary>

T9 Tastaturen funktionierten i.d.R. allerdings mit einem Unigram Modell, welches lediglich das statistisch häufigste mögliche Wort innerhalb der Zeichenkombinationen vorschlug.

</details>